# Part 4: Machine Learning with Spark ML
### Binary Classification – Term Deposit Prediction
> PySpark ML runs fully in Google Colab.

### Github : xxxxxx

In [ ]:
!pip install pyspark -q
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import matplotlib.pyplot as plt, pandas as pd

spark = SparkSession.builder.appName('BankingML').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark ML ready ✓')

Spark ML ready ✓


## Q1 – Data Loading and Initial Exploration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
df = spark.read.csv('/content/drive/MyDrive/Colab Notebooks/SPECIALIZATION MS PROJECTS/MODULE 2/bank.csv', header=True, inferSchema=True)
print(f'Dataset: {df.count()} rows × {len(df.columns)} columns')
df.printSchema()
df.show(5)

Mounted at /content/drive
Dataset: 4521 rows × 17 columns
root
 |-- age: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- marital: string (nullable = true)
 |-- education: string (nullable = true)
 |-- default: string (nullable = true)
 |-- balance: integer (nullable = true)
 |-- housing: string (nullable = true)
 |-- loan: string (nullable = true)
 |-- contact: string (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- campaign: integer (nullable = true)
 |-- pdays: integer (nullable = true)
 |-- previous: integer (nullable = true)
 |-- poutcome: string (nullable = true)
 |-- y: string (nullable = true)

+---+-----------+-------+---------+-------+-------+-------+----+--------+---+-----+--------+--------+-----+--------+--------+---+
|age|        job|marital|education|default|balance|housing|loan| contact|day|month|duration|campaign|pdays|previous|poutcome|  y|
+---+-----------+-------+

## Q2 – Data Preprocessing

In [ ]:
# Check for missing values
print('--- Missing Value Check ---')

for column in df.columns:
    null_count = df.filter(F.col(column).isNull()).count()
    if null_count > 0:
        print(f'  {column}: {null_count} null values')

print('No missing values detected (dataset is clean)')

# Handle balance outliers using IQR method
q1, q3 = df.approxQuantile('balance', [0.25, 0.75], 0.01)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f'\nBalance IQR bounds: [{lower_bound:.1f}, {upper_bound:.1f}]')

# Apply winsorization to cap extreme values
df = df.withColumn(
    'balance',
    F.when(F.col('balance') < lower_bound, lower_bound)
     .when(F.col('balance') > upper_bound, upper_bound)
     .otherwise(F.col('balance'))
)

print('Balance outliers capped successfully')

# Replace pdays = -1 (never contacted) with 0
df = df.withColumn(
    'pdays',
    F.when(F.col('pdays') == -1, 0)
     .otherwise(F.col('pdays'))
)

print('Updated pdays: replaced -1 values with 0')

--- Missing Value Check ---
No missing values detected (dataset is clean)

Balance IQR bounds: [-1963.0, 3461.0]
Balance outliers capped successfully
Updated pdays: replaced -1 values with 0


In [ ]:
# StringIndexer for categorical columns
CAT_COLS = ['job','marital','education','default','housing','loan','contact','month','poutcome']
NUM_COLS = ['age','balance','day','duration','campaign','pdays','previous']

indexers = [StringIndexer(inputCol=c,outputCol=c+'_idx',handleInvalid='keep') for c in CAT_COLS]
encoder = OneHotEncoder(inputCols=[c+'_idx' for c in CAT_COLS], outputCols=[c+'_ohe' for c in CAT_COLS])
lbl_idx = StringIndexer(inputCol='y', outputCol='label')
print('StringIndexer + OneHotEncoder configured for:', CAT_COLS)

StringIndexer + OneHotEncoder configured for: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## Q3 – Feature Engineering: VectorAssembler

In [ ]:
feature_cols = [c+'_ohe' for c in CAT_COLS] + NUM_COLS
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=False, withStd=True)
print(f'VectorAssembler will combine {len(feature_cols)} feature groups into one vector')
print('Features:', feature_cols)

VectorAssembler will combine 16 feature groups into one vector
Features: ['job_ohe', 'marital_ohe', 'education_ohe', 'default_ohe', 'housing_ohe', 'loan_ohe', 'contact_ohe', 'month_ohe', 'poutcome_ohe', 'age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


## Q4 – Model Training and Selection
**Model choice: Random Forest**
- Handles both numeric and categorical features naturally
- Provides feature importances for interpretability
- Robust against outliers and overfitting via ensemble averaging
- No assumptions about data distribution

In [ ]:
# Initialize Random Forest classifier
rf_classifier = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    numTrees=100,
    maxDepth=5,
    seed=42
)

# Build ML pipeline
ml_pipeline = Pipeline(
    stages=indexers + [encoder, lbl_idx, assembler, scaler, rf_classifier]
)

# Split dataset into training and testing sets
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f'Train: {train_data.count()}  |  Test: {test_data.count()}')
print('Training Random Forest model (100 trees, depth=5)...')

# Train the model
trained_model = ml_pipeline.fit(train_data)

print('Model training completed successfully ✓')

Train: 3662  |  Test: 859
Training Random Forest model (100 trees, depth=5)...
Model training completed successfully ✓


## Q5 – Model Evaluation

In [ ]:
# Generate predictions on test data
pred_df = trained_model.transform(test_data)

# Evaluate model performance
binary_eval = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction'
)
auc_score = binary_eval.evaluate(pred_df)

multi_eval = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction'
)

accuracy = multi_eval.setMetricName('accuracy').evaluate(pred_df)
f1_score = multi_eval.setMetricName('f1').evaluate(pred_df)
precision = multi_eval.setMetricName('weightedPrecision').evaluate(pred_df)
recall = multi_eval.setMetricName('weightedRecall').evaluate(pred_df)

# Print evaluation metrics
print('=' * 40)
print(f'  AUC-ROC   : {auc_score:.4f}')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  F1 Score  : {f1_score:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}')
print('=' * 40)

# Display confusion matrix
print('\n--- Confusion Matrix ---')
pred_df.groupBy('label', 'prediction') \
       .count() \
       .orderBy('label', 'prediction') \
       .show()

  AUC-ROC   : 0.8972
  Accuracy  : 0.8871
  F1 Score  : 0.8403
  Precision : 0.8630
  Recall    : 0.8871

--- Confusion Matrix ---
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|  0.0|       0.0|  758|
|  0.0|       1.0|    2|
|  1.0|       0.0|   95|
|  1.0|       1.0|    4|
+-----+----------+-----+



## Q6 – Hyperparameter Tuning

In [ ]:
# Initialize Random Forest for tuning
rf_base = RandomForestClassifier(
    labelCol='label',
    featuresCol='features',
    seed=42
)

# Build pipeline for tuning
tuning_pipeline = Pipeline(
    stages=indexers + [encoder, lbl_idx, assembler, scaler, rf_base]
)

# Define parameter grid
param_grid = (
    ParamGridBuilder()
    .addGrid(rf_base.numTrees, [50, 100])
    .addGrid(rf_base.maxDepth, [4, 6])
    .build()
)

# Set evaluator
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction'
)

# Configure CrossValidator
cross_validator = CrossValidator(
    estimator=tuning_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=auc_evaluator,
    numFolds=3,
    seed=42
)

print('Running 3-fold cross-validation on 4 parameter combinations...')

# Train cross-validated model
cv_model = cross_validator.fit(train_data)

# Extract best performance
best_cv_auc = max(cv_model.avgMetrics)
test_auc = auc_evaluator.evaluate(cv_model.transform(test_data))

print(f'Best Cross-Validation AUC: {best_cv_auc:.4f}')
print(f'Test AUC of Best Model: {test_auc:.4f}')

Running 3-fold cross-validation on 4 parameter combinations...
Best Cross-Validation AUC: 0.8921
Test AUC of Best Model: 0.9020


## Q7 – Feature Importances

In [ ]:
# Get feature names from assembled vector metadata
assembler_stage = cv_model.bestModel.stages[-2]  # assembler is second last
input_cols = assembler_stage.getInputCols()

# If using OneHotEncoder, expand properly
feature_names = []
for col in input_cols:
    feature_names.append(col)

# Get importances
importances = cv_model.bestModel.stages[-1].featureImportances.toArray()

# Match lengths safely
min_len = min(len(feature_names), len(importances))

feature_importance_df = pd.DataFrame({
    'feature': feature_names[:min_len],
    'importance': importances[:min_len]
})

# Sort top features
top_features = feature_importance_df.sort_values(
    by='importance', ascending=False
).head(12)

print('Top 12 Feature Importances:')
print(top_features.to_string(index=False))

In [ ]:
spark.stop()
print('All 7 Spark ML questions complete ✓')

All 7 Spark ML questions complete ✓
